# Unit Test Generator - Fine-tuning with Unsloth (Leakage-Safe ChatML Workflow)

This notebook fine-tunes a Qwen2.5-Coder model for generating Python unit tests.

Updates in this version:
- stronger data preprocessing and filtering
- strict train/validation/test separation to avoid leakage
- held-out testing with BLEU/ROUGE, syntax validity, coverage, and mutation metrics

Scope note: class imbalance handling is intentionally excluded because it is not relevant to this generation task.

 ## 1. Load Base Model

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detection
load_in_4bit = True  # Use 4bit quantization to reduce memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Explicitly ensure we are using the model's native chat template
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5", # Standard chat template used by Qwen2.5 Instruct
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/sachithra/miniconda3/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


 ## 2. Add LoRA Adapters

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0.1,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
 )

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.12.5 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## Data Preprocessing and Split Strategy

In [3]:
from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd
import hashlib

TRAIN_VAL_SEED = 42
TEST_SUBSET_SEED = 99
TEST_SAMPLE_SIZE = 100


def extract_unit_test_text(unit_tests):
    if unit_tests is None:
        return ""

    if isinstance(unit_tests, dict):
        code_value = unit_tests.get("code", "")
        if isinstance(code_value, list):
            return "\n\n".join(str(item).strip() for item in code_value if str(item).strip())
        if isinstance(code_value, str):
            return code_value.strip()
        return ""

    if isinstance(unit_tests, list):
        snippets = []
        for item in unit_tests:
            if isinstance(item, dict):
                code_value = item.get("code", "")
                if str(code_value).strip():
                    snippets.append(str(code_value).strip())
            elif isinstance(item, str) and item.strip():
                snippets.append(item.strip())
        return "\n\n".join(snippets)

    if isinstance(unit_tests, str):
        return unit_tests.strip()

    return ""


def clean_dataframe(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    working = df.copy()

    for column in ["question", "code_ground_truth", "unit_tests"]:
        if column not in working.columns:
            working[column] = ""

    working["question"] = working["question"].fillna("").astype(str).str.strip()
    working["code_ground_truth"] = working["code_ground_truth"].fillna("").astype(str).str.strip()
    working["unit_tests_text"] = working["unit_tests"].apply(extract_unit_test_text).fillna("").astype(str).str.strip()

    before = len(working)
    missing_question = working["question"].eq("")
    missing_code = working["code_ground_truth"].eq("")
    missing_tests = working["unit_tests_text"].eq("")

    filtered = working[~(missing_question | missing_code | missing_tests)].copy()
    after_filter = len(filtered)

    deduped = filtered.drop_duplicates(subset=["question", "code_ground_truth"]).reset_index(drop=True)
    after_dedup = len(deduped)

    deduped["sample_id"] = deduped.apply(
        lambda row: hashlib.sha256(
            f"{row['question']}\n<SEP>\n{row['code_ground_truth']}".encode("utf-8")
        ).hexdigest(),
        axis=1,
    )

    print(f"[{split_name}] before={before}, after_filter={after_filter}, after_dedup={after_dedup}")
    print(
        f"[{split_name}] dropped_missing_question={int(missing_question.sum())}, "
        f"dropped_missing_code={int(missing_code.sum())}, "
        f"dropped_missing_tests={int(missing_tests.sum())}, "
        f"dropped_duplicates={after_filter - after_dedup}"
    )

    return deduped[["question", "code_ground_truth", "unit_tests_text", "sample_id"]]


def print_length_stats(df: pd.DataFrame, split_name: str) -> None:
    print(f"\nLength stats for {split_name}:")
    for column in ["question", "code_ground_truth", "unit_tests_text"]:
        lengths = df[column].astype(str).str.len()
        print(
            f"- {column}: min={int(lengths.min())}, median={int(lengths.median())}, "
            f"p95={int(lengths.quantile(0.95))}, max={int(lengths.max())}"
        )


raw_train_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="train")
raw_test_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="test")

print(f"Raw train size: {len(raw_train_dataset)}")
print(f"Raw test size:  {len(raw_test_dataset)}")

df_train_raw = raw_train_dataset.to_pandas()
df_test_raw = raw_test_dataset.to_pandas()

train_clean_df = clean_dataframe(df_train_raw, "train")
test_clean_df = clean_dataframe(df_test_raw, "test")

print_length_stats(train_clean_df, "train_clean")
print_length_stats(test_clean_df, "test_clean")

train_universe_ids = set(train_clean_df["sample_id"])
test_overlap_count = int(test_clean_df["sample_id"].isin(train_universe_ids).sum())
if test_overlap_count > 0:
    test_clean_df = test_clean_df[~test_clean_df["sample_id"].isin(train_universe_ids)].reset_index(drop=True)
print(f"Removed {test_overlap_count} overlapping rows from test split.")

train_df, val_df = train_test_split(
    train_clean_df,
    test_size=0.2,
    random_state=TRAIN_VAL_SEED,
    shuffle=True,
    )

heldout_size = min(TEST_SAMPLE_SIZE, len(test_clean_df))
test_heldout_df = test_clean_df.sample(n=heldout_size, random_state=TEST_SUBSET_SEED).reset_index(drop=True)

train_ids = set(train_df["sample_id"])
val_ids = set(val_df["sample_id"])
assert train_ids.isdisjoint(val_ids), "Train and validation splits overlap."

print(
    f"Final split sizes -> train: {len(train_df)}, "
    f"validation: {len(val_df)}, held-out test: {len(test_heldout_df)}"
 )

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False)
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True), preserve_index=False)
test_dataset = Dataset.from_pandas(test_heldout_df.reset_index(drop=True), preserve_index=False)

Raw train size: 17562
Raw test size:  62900
[train] before=17562, after_filter=17562, after_dedup=17541
[train] dropped_missing_question=0, dropped_missing_code=0, dropped_missing_tests=0, dropped_duplicates=21
[test] before=62900, after_filter=62900, after_dedup=62871
[test] dropped_missing_question=0, dropped_missing_code=0, dropped_missing_tests=0, dropped_duplicates=29

Length stats for train_clean:
- question: min=29, median=1356, p95=3002, max=7122
- code_ground_truth: min=39, median=516, p95=1653, max=7094
- unit_tests_text: min=17081, median=156973, p95=228167, max=427573

Length stats for test_clean:
- question: min=26, median=416, p95=2181, max=7122
- code_ground_truth: min=24, median=429, p95=1288, max=7094
- unit_tests_text: min=9413, median=88098, p95=202947, max=461511
Removed 17541 overlapping rows from test split.
Final split sizes -> train: 14032, validation: 3509, held-out test: 100


 ## 3. Prepare Training Data

In [4]:
def formatting_prompts_func(examples):
    questions = examples["question"]
    codes = examples["code_ground_truth"]
    unit_tests_text = examples["unit_tests_text"]

    texts = []
    for question, code, full_test_suite in zip(questions, codes, unit_tests_text):
        user_content = (
            "Write a comprehensive Python unit test suite for this code.\n\n"
            f"Problem Description:\n{question}\n\n"
            f"Code to Test:\n{code}"
        )

        messages = [
            {"role": "system", "content": "You are a helpful coding assistant that writes Python unit tests."},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": str(full_test_suite).strip()},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        ).strip()
        texts.append(text)

    return {"text": texts}


train_dataset = train_dataset.map(formatting_prompts_func, batched=True, desc="Formatting train prompts")
val_dataset = val_dataset.map(formatting_prompts_func, batched=True, desc="Formatting validation prompts")
test_dataset = test_dataset.map(formatting_prompts_func, batched=True, desc="Formatting held-out test prompts")

print(
    f"Formatted dataset sizes -> train: {len(train_dataset)}, "
    f"validation: {len(val_dataset)}, held-out test: {len(test_dataset)}"
 )

Formatting held-out test prompts: 100%|██████████| 100/100 [00:00<00:00, 5937.07 examples/s]

Formatted dataset sizes -> train: 14032, validation: 3509, held-out test: 100


 ## 4. Train the Model

In [ ]:
import gc

from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback, TrainerCallback

if hasattr(model, "config"):
    # Required so gradient checkpointing can reduce activation memory.
    model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()


class CUDACacheCleanupCallback(TrainerCallback):
    def __init__(self, every_n_steps: int = 10):
        self.every_n_steps = every_n_steps

    def on_step_end(self, _args, state, control, **_kwargs):
        _ = (_args, _kwargs)
        if torch.cuda.is_available() and state.global_step > 0 and state.global_step % self.every_n_steps == 0:
            torch.cuda.empty_cache()
            gc.collect()
        return control


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        dataset_num_proc = 1,
        gradient_accumulation_steps = 8,
        gradient_checkpointing = True,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 20,
        eval_accumulation_steps = 1,
        save_strategy = "steps",
        save_steps = 20,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        prediction_loss_only = True,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        optim = "paged_adamw_8bit",
        weight_decay = 0.001,
        max_grad_norm = 1.0,
        lr_scheduler_type = "cosine",
        group_by_length = True,
        dataloader_pin_memory = False,
        dataloader_num_workers = 0,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2),
        CUDACacheCleanupCallback(every_n_steps=10),
    ],
 )

print("Trainer configured with memory-safe settings and validation split for early stopping. Held-out test remains untouched for final testing.")

[trl.trainer.sft_trainer|WARNING]You are using a per_device_train_batch_size of 1 with padding-free training. Using a batch size of 1 anihilate the benefits of padding-free training. Please consider increasing the batch size to at least 2.
Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/14032 [00:00<?, ? examples/s]

## Train model

In [ ]:
import gc

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

trainer_stats = trainer.train()

## 5. Held-Out Test Evaluation (Generative + Syntax + Coverage + Mutation)

In [ ]:
import ast
import gc
import re
import shutil
from pathlib import Path

import evaluate
from tqdm import tqdm

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

FastLanguageModel.for_inference(model)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

predictions = []
references = []
syntax_valid_flags = []
evaluation_records = []

eval_sample_size = min(100, len(test_dataset))
eval_subset = test_dataset.select(range(eval_sample_size))

print(f"Generating test cases for {eval_sample_size} held-out samples...")
for i in tqdm(range(len(eval_subset))):
    sample = eval_subset[i]

    user_content = (
        "Write a comprehensive Python unit test suite for this code.\n\n"
        f"Problem Description:\n{sample['question']}\n\n"
        f"Code to Test:\n{sample['code_ground_truth']}"
    )

    messages = [
        {"role": "system", "content": "You are a helpful coding assistant that writes Python unit tests."},
        {"role": "user", "content": user_content},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    code_blocks = re.findall(r"```(?:python)?\n(.*?)\n```", response_text, re.DOTALL)
    if code_blocks:
        response_text = "\n\n".join(code_blocks).strip()

    ground_truth = str(sample.get("unit_tests_text", "")).strip()

    predictions.append(response_text)
    references.append([ground_truth])

    try:
        ast.parse(response_text)
        syntax_valid_flags.append(True)
    except SyntaxError:
        syntax_valid_flags.append(False)

    evaluation_records.append(
        {
            "index": i,
            "sample_id": sample.get("sample_id", f"sample_{i}"),
            "subject_code": str(sample.get("code_ground_truth", "")),
            "generated_test": response_text,
        }
    )

    del outputs, inputs, new_tokens
    if torch.cuda.is_available() and (i + 1) % 10 == 0:
        torch.cuda.empty_cache()
        gc.collect()

print("\n--- Held-Out Test Evaluation Results ---")

filtered_predictions = []
filtered_references = []
empty_preds_count = 0

for pred, ref in zip(predictions, references):
    if not ref[0].strip():
        continue

    if not pred.strip():
        empty_preds_count += 1
        pred = "# empty"

    filtered_predictions.append(pred)
    filtered_references.append(ref)

if empty_preds_count > 0:
    print(
        f"Warning: The model generated empty responses for {empty_preds_count} "
        f"out of {len(filtered_references)} valid samples."
    )

if len(filtered_references) == 0:
    print("Warning: All ground truth references were empty. Cannot compute BLEU/ROUGE.")
else:
    rouge_results = rouge_metric.compute(
        predictions=filtered_predictions,
        references=[ref[0] for ref in filtered_references],
    )
    bleu_results = bleu_metric.compute(
        predictions=filtered_predictions,
        references=filtered_references,
    )

    print(f"BLEU Score:   {bleu_results['bleu'] * 100:.2f}%")
    print(f"ROUGE-1:      {rouge_results['rouge1'] * 100:.2f}%")
    print(f"ROUGE-2:      {rouge_results['rouge2'] * 100:.2f}%")
    print(f"ROUGE-L:      {rouge_results['rougeL'] * 100:.2f}%")

syntax_validity_rate = (
    sum(syntax_valid_flags) / len(syntax_valid_flags) if syntax_valid_flags else 0.0
)
print(f"Syntax Validity Rate: {syntax_validity_rate * 100:.2f}%")

coverage_sample_size = min(10, len(evaluation_records))
print(
    f"\nRunning coverage and mutation checks on {coverage_sample_size} "
    "held-out generated tests..."
 )

try:
    from benchmark.evaluation.coverage_eval import CoverageEvaluator
    from benchmark.evaluation.mutation_eval import MutationTester

    benchmark_root = Path("benchmark")
    eval_root = benchmark_root / "temp_new_notebook_eval"
    subjects_dir = eval_root / "subjects"
    tests_model_dir = eval_root / "generated_tests" / "new_model"
    results_dir = eval_root / "results"

    if eval_root.exists():
        shutil.rmtree(eval_root)

    subjects_dir.mkdir(parents=True, exist_ok=True)
    tests_model_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    def prepare_test_code(raw_code: str, module_name: str) -> str:
        code = (raw_code or "").strip()
        if not code:
            return ""

        prefix_lines = []
        if "import unittest" not in code:
            prefix_lines.append("import unittest")
        if f"from {module_name} import" not in code and f"import {module_name}" not in code:
            prefix_lines.append(f"from {module_name} import *")

        prefix = "\n".join(prefix_lines)
        if prefix:
            return f"{prefix}\n\n{code}"
        return code

    coverage_evaluator = CoverageEvaluator(
        subjects_dir="temp_new_notebook_eval/subjects",
        generated_tests_dir="temp_new_notebook_eval/generated_tests",
        results_dir="temp_new_notebook_eval/results",
    )
    mutation_tester = MutationTester(
        subjects_dir="temp_new_notebook_eval/subjects",
        generated_tests_dir="temp_new_notebook_eval/generated_tests",
        results_dir="temp_new_notebook_eval/results",
        max_mutants_per_file=20,
    )

    coverage_results = []
    mutation_results = []

    for idx, record in enumerate(evaluation_records[:coverage_sample_size]):
        module_name = f"subject_{idx}"
        subject_file = subjects_dir / f"{module_name}.py"
        test_file = tests_model_dir / f"test_{module_name}.py"

        subject_code = str(record["subject_code"]).strip()
        generated_test = prepare_test_code(str(record["generated_test"]), module_name)

        if not subject_code or not generated_test:
            continue

        subject_file.write_text(subject_code, encoding="utf-8")
        test_file.write_text(generated_test, encoding="utf-8")

        coverage_result = coverage_evaluator.measure_coverage(
            test_file,
            subject_file,
            timeout=120,
        )
        mutation_result = mutation_tester.test_mutants(test_file, subject_file)

        coverage_results.append(coverage_result)
        mutation_results.append(mutation_result)

    successful_coverage = [result for result in coverage_results if result.success]
    successful_mutation = [result for result in mutation_results if result.success]

    if successful_coverage:
        avg_statement_coverage = sum(
            result.statement_coverage for result in successful_coverage
        ) / len(successful_coverage)
        avg_branch_coverage = sum(
            result.branch_coverage for result in successful_coverage
        ) / len(successful_coverage)
        print(f"Average Statement Coverage: {avg_statement_coverage * 100:.2f}%")
        print(f"Average Branch Coverage:    {avg_branch_coverage * 100:.2f}%")
    else:
        print("Coverage evaluation did not return successful runs.")

    if successful_mutation:
        avg_mutation_score = sum(
            result.mutation_score for result in successful_mutation
        ) / len(successful_mutation)
        print(f"Average Mutation Score:    {avg_mutation_score * 100:.2f}%")
    else:
        print("Mutation evaluation did not return successful runs.")

except Exception as metric_error:
    print(f"Coverage/Mutation evaluation skipped due to environment issue: {metric_error}")

 ## 6. Save Merged Model

In [ ]:
output_dir = "merged_model_16bit_new"

if "model" not in globals():
    raise RuntimeError("Model is not available. Run training cells before saving.")

model.save_pretrained_merged(output_dir, tokenizer, save_method="merged_16bit")
print(f"Saved merged 16-bit Hugging Face model to '{output_dir}/'")